In [0]:
# Configuration
from pyspark.sql import functions as F

UTILS_DIR = "/Workspace/Repos/ts.maira@gmail.com/techmart-catalog-pipeline/utils"
exec(open(f"{UTILS_DIR}/config.py").read())

print("✅ Config loaded")

In [0]:
# Build the Gold aggregation table
# This is the business-ready table that analysts and dashboards consume
# Aggregated by Category and Sub-Category

df_gold = (
    spark.table(TAXONOMY_ENRICHED)
    .filter(F.col("judge_approved") == "True")  # ← only approved records
    .groupBy("category", "sub_category")
    .agg(
        F.count("product_id").alias("num_products"),
        F.round(F.avg("unit_price_usd"), 2).alias("avg_price_usd"),
        F.round(F.min("unit_price_usd"), 2).alias("min_price_usd"),
        F.round(F.max("unit_price_usd"), 2).alias("max_price_usd")
    )
    .orderBy("category", "sub_category")
)

print("✅ Gold aggregation complete")
display(df_gold)

In [0]:
# Write to Gold Delta table
(df_gold.write
    .format("delta")
    .mode("overwrite")
    # .option("mergeSchema", "true")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_SUMMARY))

print(f"✅ Written: {GOLD_SUMMARY}")

In [0]:
#  Validate
print("=== VALIDATION ===")
print(f"Gold summary rows: {spark.table(GOLD_SUMMARY).count()}")
display(spark.table(GOLD_SUMMARY))